# Treinamento ResNet + LSTM Multimodal — Versão GPU-Otimizada

Notebook refatorado com foco em **redução de consumo de VRAM** e throughput.

### Otimizações aplicadas
| Técnica | Impacto |
|---|---|
| `torch.amp.autocast` (AMP FP16) | ~40-50 % menos VRAM, ~2× mais rápido |
| `torch.compile` (PyTorch ≥ 2.0) | ~10-30 % ganho de velocidade |
| `gradient_checkpointing` na backbone | troca VRAM por tempo de cômputo |
| `pin_memory=False` + `num_workers=2` | menos RAM fixada pelo DataLoader |
| AudioMAE congelado fora dos trials | sem grad do encoder entre trials |
| `gc.collect + cuda.empty_cache` garantidos | sem fragmentação de memória |
| `frame_step=2` no vídeo | metade dos frames carregados |
| `epoch_frac=0.1` | 10 % dos grupos por época |
| Acumulação de gradiente (`accum_steps`) | simula batch maior sem aumentar VRAM |
| Métricas movidas para CPU imediatamente | tensores não ficam presos na VRAM |


## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys, datetime, random, gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

import optuna
from transformers import AutoModel
import einops
import timm
import torchaudio

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

from models.nn.resnetlstm import ResNetLSTM
from models.nn.resnetlstm_multimodal import ResNetLSTM_MultiModal
from preprocessing.loader_multimodal_frac2 import (
    build_multimodal_dataloader,
    train_video_transform,
    TARGET_SHAPE,
    train_mel_transform,
)
from preprocessing.balanced_dataset import balanced_df

/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    # Habilita TF32 para ops de matriz — sem perda de qualidade notável, +15 % velocidade
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  # autotuner de kernels

SEED = 435
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Torch: 2.5.1+cu121
Torchvision: 0.20.1+cu121
CUDA disponível: True
CUDA: 12.1
GPU: NVIDIA RTX A4500
Device: cuda


## 2. Configuração

In [4]:
# ── Paths ────────────────────────────────────────────────────────────────────
LABELS_ALL  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_combinadas_wg.csv"
LABELS_DIR  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used"
CKPT_DIR    = "/mnt/storage_C4/gaussian_football/models/checkpoints"
RUNS_DIR    = "/mnt/storage_C4/gaussian_football/runs_multimodal_gpu_opt"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

# ── Treino ────────────────────────────────────────────────────────────────────
EPOCHS       = 100
PATIENCE     = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
MONITOR      = "loss"   # "ccc" | "mae" | "loss"

# ── DataLoader ────────────────────────────────────────────────────────────────
BATCH_SIZE        = 2
BATCH_SIZE_RES50  = 1

# ── GPU / memória ─────────────────────────────────────────────────────────────
USE_AMP        = True   # Mixed Precision FP16 — principal economia de VRAM
USE_COMPILE    = True   # torch.compile (PyTorch >= 2.0); desative se der erro
GRAD_ACCUM     = 2      # acumula N mini-batches antes de step → simula batch*N
GRAD_CKPT      = True   # gradient checkpointing na backbone (VRAM ↓, tempo ↑)
FRAME_STEP     = 2      # lê 1 frame a cada N → metade dos frames
FRAC_EPOCH     = 0.1    # usa 10 % dos grupos por época
NUM_WORKERS    = 2      # menos workers = menos vídeos simultâneos na RAM

# ── Modelo ────────────────────────────────────────────────────────────────────
MODEL_DEFAULTS = dict(
    frame_step=FRAME_STEP,
    hidden_size=256,
    num_layers=1,
    use_dropout=False,
    dropout_p=0.3,
)

## 3. Balanceando os Dados

In [5]:
FORCE_REBUILD = True

paths_balanced = {
    "train": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_train_wg.csv"),
    "valid": os.path.join(LABELS_DIR, "balanced_todas_as_ligas_valid_wg.csv"),
    "test":  os.path.join(LABELS_DIR, "balanced_todas_as_ligas_test_wg.csv"),
}

if FORCE_REBUILD or not all(os.path.exists(p) for p in paths_balanced.values()):
    all_data = pd.read_csv(LABELS_ALL)
    for split, out_path in paths_balanced.items():
        subset  = all_data[all_data["split"] == split]
        balanced = balanced_df(subset, "game_id", threshold=0.5, random_state=SEED)
        balanced.to_csv(out_path, index=False)
        print(f"{split}: {len(balanced)} clipes -> {out_path}")
else:
    print("CSVs balanceados já existem.")

train: 16912 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_train_wg.csv
valid: 6152 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_valid_wg.csv
test: 5730 clipes -> /mnt/storage_C4/gaussian_football/data/balanced_datasets/used/balanced_todas_as_ligas_test_wg.csv


## 4. DataLoaders Multimodais

In [6]:
TRAIN_PATH = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_train_wg.csv"
VAL_PATH   = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_valid_wg.csv"
TEST_PATH  = "/mnt/storage_C4/gaussian_football/data/balanced_datasets/used/todas_as_ligas_test_wg.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

In [7]:
# Filtra apenas goal / Background
eventos_usados = ["goal", "Background"]
dir_bg = "/mnt/storage_C4/gaussian_football/data/datasets_goal_background"
os.makedirs(dir_bg, exist_ok=True)

for nome, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    filt = df[df["type"].isin(eventos_usados)]
    filt.to_csv(f"{dir_bg}/{nome}.csv", index=False)
    print(f"{nome}: {filt['type'].value_counts().to_dict()}")

TRAIN_PATH = f"{dir_bg}/train.csv"
VAL_PATH   = f"{dir_bg}/val.csv"
TEST_PATH  = f"{dir_bg}/test.csv"

train: {'Background': 56396, 'goal': 8460}
val: {'Background': 17917, 'goal': 3076}
test: {'Background': 18697, 'goal': 2865}


In [8]:
GROUPS            = True
GROUPS_COLUMN_ID  = "window_id"

def _make_loader(csv_path, split, pair, batch_size, shuffle):
    """Fábrica de DataLoader com configurações GPU-otimizadas."""
    return build_multimodal_dataloader(
        csv_path=csv_path,
        pair=pair,
        split=split,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        is_grayscale=True,
        pin_memory=False,       # True esgota RAM fixada com vídeos grandes
        groups=GROUPS,
        column_groups_id=GROUPS_COLUMN_ID,
        video_transform=train_video_transform if shuffle else None,
        mel_transform=train_mel_transform if shuffle else None,
        target_shape=TARGET_SHAPE,
        epoch_frac=FRAC_EPOCH,
        frame_step=FRAME_STEP,
    )

loaders = {
    bs: (
        _make_loader(TRAIN_PATH, "train", pair=True,  batch_size=bs, shuffle=True),
        _make_loader(VAL_PATH,   "valid", pair=True,  batch_size=bs, shuffle=False),
    )
    for bs in [1, 2]
}

Dataset de Treino: 62066/64856 exemplos válidos.
6144 grupos encontrados.
Low: 53977
High: 8089

897 pares de grupos criados.

Dataset de Validação: 18642/20993 exemplos válidos.
1827 grupos encontrados.
Low: 15961
High: 2681

295 pares de grupos criados.

Dataset de Treino: 62066/64856 exemplos válidos.
6144 grupos encontrados.
Low: 53977
High: 8089

897 pares de grupos criados.

Dataset de Validação: 18642/20993 exemplos válidos.
1827 grupos encontrados.
Low: 15961
High: 2681

295 pares de grupos criados.



## 5. Métricas

In [9]:
def calculate_ccc(y_true, y_pred):
    """Concordance Correlation Coefficient sobre dois arrays 1-D."""
    mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)
    var_true,  var_pred  = np.var(y_true),  np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    denom = var_true + var_pred + (mean_true - mean_pred) ** 2
    return 0.0 if denom == 0 else (2 * cov) / denom

def binary_accuracy(y_true, y_pred, threshold=0.5):
    return float(np.mean((y_pred > threshold) == (y_true > threshold)))

## 6. Loss Combinada (BCE + Margin Ranking Adaptativa)

In [10]:
class CombinedLoss(nn.Module):
    """BCE + Margin Ranking Loss com margem adaptativa."""

    def __init__(self, alpha=1.0, margin_scale=1.0):
        super().__init__()
        self.alpha = alpha
        self.margin_scale = margin_scale
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, low_label, high_label, low_pred, high_pred, return_components=False):
        bce = (self.bce(low_pred, low_label) + self.bce(high_pred, high_label)) / 2
        margin = self.margin_scale * (high_label - low_label)
        rank = torch.relu(margin - (high_pred - low_pred)).mean()
        loss = bce + self.alpha * rank
        return (loss, bce, rank) if return_components else loss

## 7. Treino GPU-Otimizado

In [11]:
MONITOR_MODE = {"ccc": "max", "mae": "min", "loss": "min"}


def _apply_freeze(model, freeze_backbone):
    """Liga/desliga gradiente da ResNet; conv1 (idx 0) sempre treina."""
    for name, param in model.cnn.named_parameters():
        param.requires_grad = name.startswith("0.") or (not freeze_backbone)


def _set_backbone_eval(model):
    """Congela BatchNorm na ResNet, mantém conv1 em train."""
    for i, module in enumerate(model.cnn):
        (module.train if i == 0 else module.eval)()


def _enable_gradient_checkpointing(model):
    """
    Ativa gradient checkpointing nos blocos da ResNet para trocar VRAM por
    recompute durante o backward. Compatível com timm/torchvision ResNets.
    """
    for module in model.cnn.modules():
        if hasattr(module, "gradient_checkpointing_enable"):
            module.gradient_checkpointing_enable()
        # Para ResNets do torchvision, aplica em cada BasicBlock/Bottleneck
        elif hasattr(module, "use_checkpoint"):
            module.use_checkpoint = True


def _is_better(current, best, mode):
    return current > best if mode == "max" else current < best


def _log_pred_scatter(writer, y_true, y_pred, tag, step):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(y_true, y_pred, s=8, alpha=0.4)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, "--", color="gray", linewidth=1)
    ax.set_xlabel("target"); ax.set_ylabel("pred"); ax.set_title("pred vs target")
    writer.add_figure(tag, fig, step)
    plt.close(fig)


def train_model(
    model, train_loader, val_loader, *,
    criterion, run_name, optimizer, scheduler,
    backbone_name, loss_name,
    freeze_mode="finetune", unfreeze_epoch=5,
    freeze_bn_always=True,
    epochs=EPOCHS, patience=PATIENCE, monitor=MONITOR,
    grad_clip=GRAD_CLIP,
    use_amp=USE_AMP,
    accum_steps=GRAD_ACCUM,
    checkpoint_path=None, device=device,
    trial=None,
    lr=LR,
):
    """
    Treina e valida por época com Mixed Precision + acumulação de gradiente.

    Parâmetros extras vs versão original:
        use_amp:      Mixed Precision FP16 (recomendado True).
        accum_steps:  Acumula N mini-batches antes do step do optimizer.
                      Simula batch_size * accum_steps sem aumentar VRAM.
    """
    model.to(device)
    gc.collect()
    torch.cuda.empty_cache()

    if checkpoint_path is None:
        checkpoint_path = os.path.join(CKPT_DIR, f"{run_name}.pth")
    last_path = os.path.join(CKPT_DIR, f"{run_name}_last.pth")

    mode = MONITOR_MODE[monitor]
    best_score = -np.inf if mode == "max" else np.inf
    best_metrics, best_true, best_pred = {}, None, None
    epochs_no_improve = 0

    # GradScaler só é relevante com AMP
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    stamp   = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(RUNS_DIR, f"{run_name}_{stamp}")
    writer  = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):
        # ── Freeze schedule ──────────────────────────────────────────────────
        if freeze_mode == "full":
            backbone_frozen = False
        elif freeze_mode == "frozen":
            backbone_frozen = True
        else:  # finetune
            backbone_frozen = epoch < unfreeze_epoch
            if epoch == unfreeze_epoch:
                print(f"[época {epoch+1}] descongelando backbone (fine-tuning completo)")

        _apply_freeze(model, backbone_frozen)

        # ── Treino ───────────────────────────────────────────────────────────
        model.train()
        if freeze_bn_always or backbone_frozen:
            _set_backbone_eval(model)

        train_loss = train_bce = train_rank = 0.0
        train_true_buf, train_pred_buf = [], []
        optimizer.zero_grad(set_to_none=True)  # set_to_none economiza memória

        for step, (low, high) in enumerate(
            tqdm(train_loader, desc=f"época {epoch+1}/{epochs} [treino]", leave=False)
        ):
            low_video,  _, low_mel,  low_targets  = low
            high_video, _, high_mel, high_targets = high

            # Transferência assíncrona (non_blocking=True): CPU não espera GPU
            low_video   = low_video.to(device,  non_blocking=True)
            high_video  = high_video.to(device, non_blocking=True)
            low_mel     = low_mel.to(device,    non_blocking=True)
            high_mel    = high_mel.to(device,   non_blocking=True)
            low_targets = low_targets.to(device).float().view(-1)
            high_targets= high_targets.to(device).float().view(-1)

            with torch.amp.autocast("cuda", enabled=use_amp):
                low_out  = model(low_video,  low_mel).reshape(-1)
                high_out = model(high_video, high_mel).reshape(-1)
                loss, bce_l, rank_l = criterion(
                    low_targets, high_targets, low_out, high_out,
                    return_components=True,
                )
                # Normaliza pela acumulação para gradientes comparáveis
                loss = loss / accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    grad_clip,
                )
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            bs = low_video.size(0)
            # Detach + cpu imediato libera VRAM do tensor de saída
            preds   = torch.cat([torch.sigmoid(low_out), torch.sigmoid(high_out)]).detach().cpu()
            targets = torch.cat([low_targets, high_targets]).detach().cpu()

            train_loss += loss.item() * accum_steps * bs
            train_bce  += bce_l.item() * bs
            train_rank += rank_l.item() * bs

            train_true_buf.extend(targets.numpy())
            train_pred_buf.extend(preds.numpy())

        # Passo residual se o dataset não for múltiplo de accum_steps
        if (len(train_loader) % accum_steps) != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), grad_clip
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        n = len(train_loader.dataset)
        train_loss /= n; train_bce /= n; train_rank /= n

        train_true = np.array(train_true_buf)
        train_pred = np.array(train_pred_buf)
        train_mae  = np.mean(np.abs(train_true - train_pred))
        train_ccc  = calculate_ccc(train_true, train_pred)
        train_acc  = binary_accuracy(train_true, train_pred)

        # ── Validação ────────────────────────────────────────────────────────
        model.eval()
        val_loss = val_bce = val_rank = 0.0
        all_true, all_pred = [], []

        with torch.no_grad():
            for (low, high) in tqdm(val_loader, desc=f"época {epoch+1}/{epochs} [val]", leave=False):
                low_video,  _, low_mel,  low_targets  = low
                high_video, _, high_mel, high_targets = high

                low_video   = low_video.to(device,  non_blocking=True)
                high_video  = high_video.to(device, non_blocking=True)
                low_mel     = low_mel.to(device,    non_blocking=True)
                high_mel    = high_mel.to(device,   non_blocking=True)
                low_targets = low_targets.to(device).float().view(-1)
                high_targets= high_targets.to(device).float().view(-1)

                with torch.amp.autocast("cuda", enabled=use_amp):
                    low_out  = model(low_video,  low_mel).reshape(-1)
                    high_out = model(high_video, high_mel).reshape(-1)
                    loss, bce_l, rank_l = criterion(
                        low_targets, high_targets, low_out, high_out,
                        return_components=True,
                    )

                bs = low_video.size(0)
                val_loss += loss.item() * bs
                val_bce  += bce_l.item() * bs
                val_rank += rank_l.item() * bs

                preds   = torch.cat([torch.sigmoid(low_out), torch.sigmoid(high_out)]).cpu()
                targets = torch.cat([low_targets, high_targets]).cpu()
                all_true.extend(targets.numpy())
                all_pred.extend(preds.numpy())

        n = len(val_loader.dataset)
        val_loss /= n; val_bce /= n; val_rank /= n

        all_true = np.array(all_true)
        all_pred = np.array(all_pred)
        val_mae  = np.mean(np.abs(all_true - all_pred))
        val_ccc  = calculate_ccc(all_true, all_pred)
        val_acc  = binary_accuracy(all_true, all_pred)

        scheduler.step(val_loss)

        # ── TensorBoard ──────────────────────────────────────────────────────
        writer.add_scalars("loss",      {"train": train_loss, "val": val_loss},  epoch)
        writer.add_scalars("bce",       {"train": train_bce,  "val": val_bce},   epoch)
        writer.add_scalars("rank_loss", {"train": train_rank, "val": val_rank},  epoch)
        writer.add_scalars("ccc",       {"train": train_ccc,  "val": val_ccc},   epoch)
        writer.add_scalars("mae",       {"train": train_mae,  "val": val_mae},   epoch)
        writer.add_scalars("acc",       {"train": train_acc,  "val": val_acc},   epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"[{epoch+1:03d}] loss={val_loss:.4f} ccc={val_ccc:.4f} "
            f"mae={val_mae:.4f} acc={val_acc:.3f}"
        )

        # ── Checkpoint / early-stopping ───────────────────────────────────────
        current_score = {"ccc": val_ccc, "mae": val_mae, "loss": val_loss}[monitor]
        torch.save(model.state_dict(), last_path)

        if _is_better(current_score, best_score, mode):
            best_score   = current_score
            best_metrics = {
                "val_ccc": val_ccc, "val_mae": val_mae,
                "val_loss": val_loss, "val_acc": val_acc,
                "val_bce": val_bce, "val_rank": val_rank,
            }
            best_true, best_pred = all_true, all_pred
            torch.save(model.state_dict(), checkpoint_path)
            epochs_no_improve = 0
            print(f"  ✓ novo melhor {monitor}: {best_score:.4f} → salvo em {checkpoint_path}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping na época {epoch+1}")
                break

        # Optuna pruning
        if trial is not None:
            trial.report(current_score, epoch)
            if trial.should_prune():
                writer.close()
                raise optuna.TrialPruned()

        # Libera tensores intermediários que possam ter ficado no grafo
        gc.collect()

    if best_true is not None:
        _log_pred_scatter(writer, best_true, best_pred, "best_pred_scatter", 0)

    writer.add_hparams(
        {
            "backbone": backbone_name, "lr": lr,
            "loss": loss_name, "monitor": monitor,
        },
        {
            "best/val_loss": best_metrics.get("val_loss", np.inf),
            "best/val_ccc":  best_metrics.get("val_ccc",  0.0),
            "best/val_acc":  best_metrics.get("val_acc",  0.0),
        },
        run_name=".",
    )
    writer.close()
    torch.cuda.empty_cache()
    print(f"Concluído. Melhor {monitor}: {best_score:.4f}")
    return {"checkpoint": checkpoint_path, "best_metrics": best_metrics}

## 8. Função `run_experiment`

In [12]:
def run_experiment(
    audiomae,
    alpha=1.0,
    margin_scale=1.0,
    backbone_name="resnet18",
    freeze_mode="finetune",
    unfreeze_epoch=5,
    lr=LR,
    lr_backbone=None,
    weight_decay=WEIGHT_DECAY,
    model_kwargs=None,
    criterion=None,
    epochs=EPOCHS,
    patience=PATIENCE,
    use_amp=USE_AMP,
    accum_steps=GRAD_ACCUM,
    use_fusion=True,
    always_bn=False,
    train_loader=None,
    val_loader=None,
    trial=None,
):
    model_kwargs = {**MODEL_DEFAULTS, **(model_kwargs or {})}
    run_name = (
        f"{backbone_name}__{freeze_mode}__fusion{use_fusion}"
        f"__bn{always_bn}__trial{trial.number if trial else 'manual'}"
    )
    print(f"\n=== {run_name} ===")

    model = ResNetLSTM_MultiModal(
        audiomae,
        backbone_name=backbone_name,
        use_fusion=use_fusion,
        **model_kwargs,
    ).to(device)

    # Gradient checkpointing na backbone — troca VRAM por recompute
    if GRAD_CKPT:
        _enable_gradient_checkpointing(model)

    # torch.compile — funde ops e elimina overhead Python no forward
    if USE_COMPILE and hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="reduce-overhead")
        except Exception as e:
            print(f"[AVISO] torch.compile falhou ({e}); usando modelo sem compilação.")

    if lr_backbone is None:
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = AdamW(
            [
                {"params": model.cnn.parameters(),  "lr": lr_backbone},
                {"params": model.lstm.parameters(), "lr": lr},
                {"params": model.head.parameters(), "lr": lr},
            ],
            weight_decay=weight_decay,
        )

    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    if criterion is None:
        criterion = CombinedLoss(alpha=alpha, margin_scale=margin_scale)

    try:
        result = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            run_name=run_name,
            optimizer=optimizer,
            scheduler=scheduler,
            backbone_name=backbone_name,
            loss_name=criterion.__class__.__name__,
            freeze_mode=freeze_mode,
            unfreeze_epoch=unfreeze_epoch,
            epochs=epochs,
            patience=patience,
            use_amp=use_amp,
            accum_steps=accum_steps,
            freeze_bn_always=always_bn,
            lr=lr,
            trial=trial,
        )
    finally:
        # Garantia de limpeza mesmo em caso de exceção
        del model, optimizer, scheduler, criterion
        gc.collect()
        torch.cuda.empty_cache()

    return result

## 9. AudioMAE — carrega e congela uma vez

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_ae = AutoModel.from_pretrained("hance-ai/audiomae", trust_remote_code=True).to(device)

# Congela o AudioMAE permanentemente — não requer grad em nenhum trial
model_ae.eval()
for p in model_ae.parameters():
    p.requires_grad = False

# Confirma memória disponível antes de começar os trials
if torch.cuda.is_available():
    free_mb = (torch.cuda.get_device_properties(0).total_memory
               - torch.cuda.memory_allocated()) / 1024**2
    print(f"VRAM livre antes dos trials: {free_mb:.0f} MB")

A new version of the following files was downloaded from https://huggingface.co/hance-ai/audiomae:
- model.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


VRAM livre antes dos trials: 19851 MB


## 10. Busca Optuna com Pruning Hyperband

In [14]:
SEARCH_EPOCHS = 40


def objective(trial):
    backbone    = trial.suggest_categorical("backbone",    ["resnet18", "resnet34", "resnet50"])
    freeze_mode = trial.suggest_categorical("freeze_mode", ["frozen", "finetune"])
    use_fusion  = trial.suggest_categorical("use_fusion",  [True, False])
    always_bn   = trial.suggest_categorical("always_bn",   [True, False])
    lr          = trial.suggest_float("lr",           1e-5, 1e-3, log=True)
    alpha       = trial.suggest_float("alpha",        0.0, 1.0)
    margin_scale= trial.suggest_float("margin_scale", 0.5, 2.0)

    bs = 1 if backbone == "resnet50" else 2
    train_loader, val_loader = loaders[bs]

    try:
        result = run_experiment(
            audiomae=model_ae,
            backbone_name=backbone,
            freeze_mode=freeze_mode,
            unfreeze_epoch=5,
            alpha=alpha,
            margin_scale=margin_scale,
            use_fusion=use_fusion,
            always_bn=always_bn,
            lr=lr,
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=SEARCH_EPOCHS,
            trial=trial,
        )
    except optuna.TrialPruned:
        raise
    except Exception as e:
        print(f"ERRO no trial {trial.number}: {e}")
        gc.collect()
        torch.cuda.empty_cache()
        return float("inf")

    return result["best_metrics"]["val_rank"]


study = optuna.create_study(
    direction="minimize",
    study_name="multimodal_gpu_opt",
    storage="sqlite:///optuna_multimodal_gpu_opt.db",
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.HyperbandPruner(min_resource=5, max_resource=SEARCH_EPOCHS),
)

study.optimize(objective, n_trials=20)

print("\n=== MELHOR ===")
print(study.best_params)
print(f"val_rank: {study.best_value:.4f}")

[I 2026-06-29 13:58:38,419] Using an existing study with name 'multimodal_gpu_opt' instead of creating a new one.



=== resnet18__finetune__fusionFalse__bnFalse__trial62 ===


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/al.pedro.santos/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 74.6MB/s]


TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_gpu_opt/resnet18__finetune__fusionFalse__bnFalse__trial62_20260629-135843


época 1/40 [treino]:  62%|██████▏   | 28/45 [02:22<00:41,  2.46s/it]moov atom not found


...

ERRO no trial 88: CUDA out of memory. Tried to allocate 564.00 MiB. GPU 0 has a total capacity of 19.71 GiB of which 539.38 MiB is free. Including non-PyTorch memory, this process has 18.72 GiB memory in use. Of the allocated memory 17.14 GiB is allocated by PyTorch, with 15.78 GiB allocated in private pools (e.g., CUDA Graphs), and 57.00 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


[I 2026-06-29 14:23:19,873] Trial 88 finished with value: inf and parameters: {'backbone': 'resnet50', 'freeze_mode': 'finetune', 'use_fusion': True, 'always_bn': False, 'lr': 6.510920985224561e-05, 'alpha': 0.36294669176169947, 'margin_scale': 1.7178765884052138}. Best is trial 1 with value: inf.



=== resnet18__finetune__fusionFalse__bnFalse__trial90 ===
TensorBoard: /mnt/storage_C4/gaussian_football/runs_multimodal_gpu_opt/resnet18__finetune__fusionFalse__bnFalse__trial90_20260629-142321


[I 2026-06-29 14:23:40,251] Trial 90 finished with value: inf and parameters: {'backbone': 'resnet18', 'freeze_mode': 'finetune', 'use_fusion': False, 'always_bn': False, 'lr': 1.4571295328157744e-05, 'alpha': 0.9363412688081247, 'margin_scale': 1.5349889892424187}. Best is trial 1 with value: inf.


ERRO no trial 90: CUDA out of memory. Tried to allocate 1.16 GiB. GPU 0 has a total capacity of 19.71 GiB of which 736.31 MiB is free. Including non-PyTorch memory, this process has 18.53 GiB memory in use. Of the allocated memory 16.95 GiB is allocated by PyTorch, with 15.78 GiB allocated in private pools (e.g., CUDA Graphs), and 46.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

=== MELHOR ===
{'backbone': 'resnet18', 'freeze_mode': 'finetune', 'use_fusion': False, 'always_bn': True, 'lr': 0.0004504191333829785, 'alpha': 0.9000474643766353, 'margin_scale': 1.376418058424284}
val_rank: inf


## 11. Relatório dos Resultados

In [15]:
import optuna.visualization as vis

# Top-5 trials
df_trials = study.trials_dataframe()
print(df_trials.sort_values("value").head(5).to_string(index=False))

# Importância dos hiperparâmetros
try:
    importances = optuna.importance.get_param_importances(study)
    for param, imp in importances.items():
        print(f"  {param:<20s}: {imp:.3f}")
except Exception as e:
    print(f"Importâncias indisponíveis: {e}")

 number    value             datetime_start          datetime_complete               duration  params_alpha  params_always_bn params_backbone params_freeze_mode  params_lr  params_margin_scale  params_use_fusion  system_attrs_completed_rung_0  system_attrs_completed_rung_1    state
      4 1.569648 2026-06-29 12:23:37.647128 2026-06-29 12:46:39.130747 0 days 00:23:01.483619      0.900047              True        resnet18           finetune   0.000450             1.376418              False                       1.569648                            NaN   PRUNED
      1      inf 2026-06-29 12:04:48.173494 2026-06-29 13:19:43.967324 0 days 01:14:55.793830      0.900047              True        resnet18           finetune   0.000450             1.376418              False                       1.569128                       1.570926 COMPLETE
      2      inf 2026-06-29 12:17:16.432708 2026-06-29 12:22:32.355208 0 days 00:05:15.922500      0.900047              True        resnet18          